# Сильный режим GNN — обучение лестницы L3→L6 на Colab A100

Считает 4 «сильных» шага лестницы перехода (PNA / Multi-PNA+EU, большие окрестности)
на полном IBM AML HI-Small и выгружает результаты zip-архивом. L0–L2 (слабый
режим) уже посчитаны — здесь только L3–L6.

**Перед запуском:**
1. `Runtime → Change runtime type → A100` (желательно **High-RAM**).
2. Данные на Google Drive: `HI-Small_Trans.csv` и `HI-Small_Patterns.txt` (не в git).

Запускай ячейки сверху вниз.

## 1. Проверка GPU

In [1]:
import torch
assert torch.cuda.is_available(), "Нет GPU! Runtime → Change runtime type → A100 (High-RAM)"
name = torch.cuda.get_device_name(0)
print("GPU:", name)
if "A100" not in name:
    print("\n⚠️  Не A100. Конфиги L3–L6 рассчитаны на A100 40GB ([50,50,50]/batch 4096).")
    print("    На L4/T4 снизь num_neighbors→[25,25,25] и batch_size→1024 ОДИНАКОВО по всем L3–L6,")
    print("    иначе прирост нельзя честно приписать рычагу.")

GPU: NVIDIA A100-SXM4-80GB


## 2. Клонирование репозитория

In [2]:
import os
REPO = "/content/gnn-aml-graphs-kr"
if not os.path.exists(REPO):
    !git clone https://github.com/Ezzenin/gnn-aml-graphs-kr.git {REPO}
%cd {REPO}
!git pull --quiet
print("repo:", REPO)

Cloning into '/content/gnn-aml-graphs-kr'...
remote: Enumerating objects: 214, done.
remote: Counting objects: 100% (214/214), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 214 (delta 102), reused 189 (delta 77), pack-reused 0 (from 0)
Receiving objects: 100% (214/214), 1.35 MiB | 41.99 MiB/s, done.
Resolving deltas: 100% (102/102), done.
/content/gnn-aml-graphs-kr
repo: /content/gnn-aml-graphs-kr


## 3. Зависимости

In [3]:
import importlib, re, subprocess, sys
import torch

base_ver = re.match(r"^(\d+\.\d+)", torch.__version__).group(1) + ".0"
cuda = torch.version.cuda
cuda_tag = "cpu" if cuda is None else "cu" + "".join(cuda.split(".")[:2])
pyg_wheels = f"https://data.pyg.org/whl/torch-{base_ver}+{cuda_tag}.html"
print("torch", torch.__version__, "| cuda", cuda, "| PyG wheels", pyg_wheels)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
                "torch_geometric", "xgboost", "networkx", "pyyaml", "wandb"], check=True)

errors = []
backend_ok = False
for package, module in [("pyg_lib", "pyg_lib"), ("torch_sparse", "torch_sparse")]:
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--no-cache-dir",
                        package, "-f", pyg_wheels], check=True)
        importlib.invalidate_caches()
        importlib.import_module(module)
    except Exception as exc:
        errors.append(f"{package}: {type(exc).__name__}: {exc}")
        continue
    backend_ok = True
    print("PyG sampler backend:", module)
    break

import torch_geometric
print("pyg", torch_geometric.__version__)
assert backend_ok, "Не установился backend для LinkNeighborLoader/NeighborSampler: " + "; ".join(errors)

torch 2.11.0+cu128 | cuda 12.8 | PyG wheels https://data.pyg.org/whl/torch-2.11.0+cu128.html
PyG sampler backend: pyg_lib
pyg 2.8.0


## 4. Данные с Google Drive
Поправь `DRIVE_DATA`, если файлы лежат в другой папке Drive.

In [4]:
from google.colab import drive
drive.mount("/content/drive")

import os, shutil
DRIVE_DATA = "/content/drive/MyDrive/data for models"   # ← папка на Drive с HI-Small_*.csv/.txt
os.makedirs(f"{REPO}/data/ibm_aml", exist_ok=True)
for fn in ["HI-Small_Trans.csv", "HI-Small_Patterns.txt"]:
    src = os.path.join(DRIVE_DATA, fn)
    assert os.path.exists(src), f"Нет файла {src} — проверь путь DRIVE_DATA"
    shutil.copy(src, f"{REPO}/data/ibm_aml/{fn}")
print("Данные на месте:", os.listdir(f"{REPO}/data/ibm_aml"))

Mounted at /content/drive
Данные на месте: ['HI-Small_Patterns.txt', 'HI-Small_Trans.csv']


## 5. SMOKE — быстрый чек, что PNA-путь не падает
Подвыборка + урезанные окрестности, 2 эпохи. Это НЕ финальные числа — только проверка
VRAM/RAM/кода перед многочасовым полным прогоном.

In [5]:
import yaml, sys, subprocess
base = yaml.safe_load(open(f"{REPO}/configs/ibm_pna_fulldata.yaml"))
base["dataset"]["max_rows"] = 500000
base["train"]["num_neighbors"] = [25, 25, 25]
base["train"]["batch_size"] = 1024
base["train"]["epochs"] = 2
base["train"]["patience"] = 2
base["output_name"] = "ibm_pna_smoke"
base["save_checkpoint"] = False
yaml.safe_dump(base, open(f"{REPO}/configs/_pna_smoke.yaml","w"), allow_unicode=True)
rc = subprocess.run([sys.executable,"-m","src.train_edge","--config","configs/_pna_smoke.yaml"], cwd=REPO).returncode
print("\nSMOKE rc =", rc, "(0 = ок, можно гнать полные прогоны ниже)")


SMOKE rc = 0 (0 = ок, можно гнать полные прогоны ниже)


## 6. Полные прогоны L3 → L6
~часы на A100. Если конфиг падает по OOM — снизь `batch_size`/`num_neighbors`
**одинаково во всех 4 конфигах** (`configs/ibm_*pna*_fulldata.yaml`,
`configs/ibm_multignn_big_fulldata.yaml`) и перезапусти ячейку.

In [ ]:
import sys, subprocess
LADDER = ["ibm_multignn_big_fulldata",  # L3
          "ibm_pna_fulldata",            # L4
          "ibm_multipna_fulldata",       # L5
          "ibm_multipna_eu_fulldata"]    # L6 (целевой)
for c in LADDER:
    print(f"\n========== {c} ==========")
    rc = subprocess.run([sys.executable,"-m","src.train_edge","--config",f"configs/{c}.yaml"], cwd=REPO).returncode
    if rc != 0:
        print(f"\n!! {c} упал (rc={rc}). Останавливаюсь.")
        print("   OOM → снизь batch_size/num_neighbors ОДИНАКОВО по L3–L6 и перезапусти с этого конфига.")
        break
else:
    print("\n✅ Все 4 прогона завершены.")


========== ibm_multignn_big_fulldata ==========

========== ibm_pna_fulldata ==========

========== ibm_multipna_fulldata ==========


## 7. (опционально) Репродукция под протоколом Egressy 60/20/20
Для внешней сопоставимости с литературой. Можно пропустить.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","src.train_edge","--config","configs/ibm_multipna_eu_egressy.yaml"], cwd=REPO)

## 8. Пересборка лестницы (таблица + фигура)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable,"-m","src.compare","--ibm"], cwd=REPO)
print("\n--- ibm_ladder.md ---")
print(open(f"{REPO}/results/ibm_ladder.md").read()[:1200])

## 9. Выгрузка результатов zip-архивом
Без токенов и git: архив создаётся в Colab и скачивается через браузер.

In [ ]:
# Выгрузка результатов (без токенов и git): zip → скачивание в браузер.
import os, zipfile
from google.colab import files

subprocess_names = ["ibm_multignn_big_fulldata", "ibm_pna_fulldata", "ibm_multipna_fulldata",
                    "ibm_multipna_eu_fulldata", "ibm_multipna_eu_egressy"]
out = "/content/strong_gnn_results.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for n in subprocess_names:
        for ext in ["_metrics.json", "_pr_curve.png"]:
            p = f"{REPO}/results/{n}{ext}"
            if os.path.exists(p):
                z.write(p, arcname=f"results/{n}{ext}")
print("В архиве:", zipfile.ZipFile(out).namelist())
files.download(out)   # упадёт в ~/Downloads на Mac